# AI Dustbin: Teacher (v2, pipeline version)
All logic lives in `teacher/teacher.py`; this notebook only drives it. It uses Sian's `crop_to_item()` and
preprocessing from `student/common.py`, so teacher and student see the same crops.

**Before Run all** (Runtime → T4 GPU):
1. Files panel → make `/content/zips` → drag in `dataset_C_v2.zip` and `dataset_D_v2_balanced.zip`
   (later also `dataset_bin.zip` and `feedback.zip` when they exist).
2. Have a GitHub token ready (repo read access) — the first cell clones the repo.

Outputs (`/content/teacher_out`, downloaded as a zip at the end):
| File | Format | For |
|---|---|---|
| `teacher_v1_gray.joblib`, `teacher_v1_plate.joblib` | 6 | repo / release |
| `soft_labels_<dataset>_<version>.csv` | 5 (out-of-fold for train) | Sian (distillation) |
| `results_teacher_v1.csv`, `results_hard_eval.csv` | — | paper / PR |

In [ ]:
# ---- 0. Code: clone the repo (teacher branch) and get student/common.py ----
import os, getpass, subprocess, sys
REPO = '/content/ai-dustbin-data'
BRANCH = 'teacher'
if not os.path.exists(REPO):
    tok = getpass.getpass('GitHub token: ')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    f'https://{tok}@github.com/Jhanviiigit/ai-dustbin-data.git', REPO], check=True)
    del tok
if not os.path.exists(f'{REPO}/student/common.py'):        # until student is merged into main
    os.makedirs(f'{REPO}/student', exist_ok=True)
    subprocess.run(f'cd {REPO} && git fetch -q origin student && git show origin/student:student/common.py > student/common.py',
                   shell=True, check=True)
    open(f'{REPO}/student/__init__.py', 'a').close()
sys.path.insert(0, REPO)
import importlib, teacher.teacher as tt
importlib.reload(tt)
print('teacher.py loaded; shared constants from:', tt.COMMON_SOURCE)
assert tt.COMMON_SOURCE == 'student/common.py', 'student/common.py not importable: crops may drift from the student'

In [ ]:
# ---- 1. Data: unzip whatever is in /content/zips ----
import glob, zipfile
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
DATA, OUT = Path('/content/data'), Path('/content/teacher_out')
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
for z in sorted(glob.glob('/content/zips/*.zip')):
    dest = DATA / Path(z).stem
    if not dest.exists():
        zipfile.ZipFile(z).extractall(dest)
print('found:', sorted(p.name for p in DATA.iterdir()))

VERSION = 'teacher_v1'
MASKS = tt.find_masks_dir(DATA / 'dataset_D_v2_balanced')
assert MASKS is not None, 'no masks/ folder inside dataset_D_v2_balanced'
VARIANTS = {   # same names as student/common.py VARIANTS
    'plate': ('dataset_C_v2', DATA / 'dataset_C_v2'),
    'gray':  ('dataset_D_v2_balanced', DATA / 'dataset_D_v2_balanced'),
}
INDEX = {}
for var, (name, root) in VARIANTS.items():
    df = tt.index_split_dataset(root, name, MASKS, crop=True)
    print(df.groupby(['split', 'class_folder', 'source']).size().unstack(fill_value=0), '\n')
    unknown = (df.source == 'unknown').mean()
    assert unknown < 0.05, f'{name}: {unknown:.0%} of images have no source; check manifest new_path'
    INDEX[var] = df

In [ ]:
# ---- 2. Look at the crops (should match the detector: item box + 10% padding) ----
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 6, figsize=(16, 6))
for r, var in enumerate(VARIANTS):
    sample = INDEX[var][INDEX[var].split == 'test'].sample(6, random_state=1)
    for c, row in enumerate(sample.to_dict('records')):
        ax[r, c].imshow(tt.load_item_image(row)); ax[r, c].set_title(f"{var}: {row['class_folder']}", fontsize=9)
        ax[r, c].axis('off')
plt.tight_layout(); plt.show()

## 3. Features
DINOv2 ViT-B/14, frozen. Train gets view 0 (detector framing) + 2 training views (random pad 0.05–0.30 and
Sian's augmentation). val/test: view 0 only. Cached in `teacher_out/cache`, so reruns are fast.

In [ ]:
ext = tt.FeatureExtractor(cache_dir=OUT / 'cache')
N_AUG = 2
F = {}
for var, df in INDEX.items():
    tr, va, te = (df[df.split == s].reset_index(drop=True) for s in ('train', 'val', 'test'))
    X, y, g, v = tt.build_training_views(ext, tr, N_AUG)
    F[var] = dict(tr=tr, va=va, te=te, X=X, y=y, g=g, v=v, Xv=ext.features(va), Xt=ext.features(te))
    print(var, 'train views', X.shape, 'val', F[var]['Xv'].shape, 'test', F[var]['Xt'].shape)

## 4. Fit one teacher per background, evaluate on test (95% Wilson CIs), save (format 6)

In [ ]:
TEACHERS, RESULTS = {}, []
for var, f in F.items():
    t = tt.Teacher(f'{VERSION}_{var}').fit(f['X'], f['y'], f['Xv'], f['va'].y.values,
                                           trained_on={'datasets': [VARIANTS[var][0]], 'n_train': len(f['tr'])})
    TEACHERS[var] = t
    t.save(OUT / f'{t.version}.joblib')
    z = t.logit(f['Xt'])
    RESULTS.append({'trained_on': var, 'tested_on': var, **tt.evaluate('uncalibrated', tt.sigmoid(z), f['te'])})
    RESULTS.append({'trained_on': var, 'tested_on': var, **tt.evaluate('calibrated', t.wet_prob(f['Xt']), f['te']),
                    'C': t.C, 'T': round(t.temperature, 3)})
res = pd.DataFrame(RESULTS); res.to_csv(OUT / f'results_{VERSION}.csv', index=False); res

## 5. Harder evaluation
* **Cross-source:** fit on RealWaste only → test on Roboflow, and back.
* **Cross-background:** plate-trained teacher on gray test images, and gray-trained on plate.

In [ ]:
HARD = []
for var, f in F.items():
    srcs = sorted(s for s in f['tr'].source.unique() if s != 'unknown')
    src_views = np.tile(f['tr'].source.values, N_AUG + 1)
    for a in srcs:
        for b in srcs:
            if a == b: continue
            mtr, mva, mte = src_views == a, (f['va'].source == a).values, (f['te'].source == b).values
            t = tt.Teacher('xsrc').fit(f['X'][mtr], f['y'][mtr], f['Xv'][mva], f['va'].y.values[mva])
            HARD.append({'eval': f'{var}: {a} → {b}', **tt.evaluate('cross-source', t.wet_prob(f['Xt'][mte]), f['te'][mte])})
for a in F:
    for b in F:
        if a != b:
            HARD.append({'eval': f'{a}-trained → {b} test', **tt.evaluate('cross-background', TEACHERS[a].wet_prob(F[b]['Xt']), F[b]['te'])})
hard = pd.DataFrame(HARD); hard.to_csv(OUT / 'results_hard_eval.csv', index=False); hard

## 6. Soft labels for Sian (format 5)
Train rows: **out-of-fold** (5 folds by group, so near-duplicates and augmented views never score themselves).
val/test rows: the full teacher. Columns: `split, class_folder, filename, teacher_wet_prob, teacher_logit, teacher_version`.
`teacher_logit` is the raw log-odds of wet; `teacher_wet_prob = sigmoid(logit / T)`. For distillation at
temperature τ, use `sigmoid(teacher_logit / τ)`.

In [ ]:
for var, f in F.items():
    t = TEACHERS[var]
    z_oof = tt.out_of_fold_logits(f['X'], f['y'], f['g'], f['v'], t.C)
    table = pd.concat([tt.soft_label_table(f['tr'], z_oof, t.temperature, t.version),
                       tt.soft_label_table(f['va'], t.logit(f['Xv']), t.temperature, t.version),
                       tt.soft_label_table(f['te'], t.logit(f['Xt']), t.temperature, t.version)])
    path = tt.write_soft_labels(table, OUT / f'soft_labels_{VARIANTS[var][0]}_{t.version}.csv')
    p = table[table.split == 'train'].teacher_wet_prob
    oof_acc = ((z_oof >= 0).astype(int) == f['tr'].y.values).mean()
    print(f'{path.name}: {len(table)} rows | train OOF acc {oof_acc:.4f} | train >0.99 conf {(np.maximum(p, 1-p) > .99).mean():.3f}')

## 7. Interface check
The teacher takes the same input as the student's `classify_image()`: one BGR crop. Saved teachers must reload.

In [ ]:
import cv2
for var in VARIANTS:
    t = tt.Teacher.load(OUT / f'{VERSION}_{var}.joblib'); t.extractor = ext
    row = F[var]['te'].iloc[0].to_dict()
    crop_bgr = np.asarray(tt.load_item_image(row))[:, :, ::-1]
    out = t.classify_image(crop_bgr)
    print(var, 'true:', tt.LABELS[row['y']], '->', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in out.items()})
    assert out['label'] in tt.LABELS

## 8. Update cycle (once bin data / feedback exist)
Fits on C/D + bin dataset + feedback, chooses C and T on all val splits, writes format 5 for every dataset
(+ `soft_labels_feedback_<version>.csv`) and the format-6 teacher. Same thing from a terminal:
`python teacher/teacher.py --version teacher_v2 --gray ... --bin ... --feedback ...`

In [ ]:
BIN, FEEDBACK = DATA / 'dataset_bin', DATA / 'feedback'
if BIN.exists() or FEEDBACK.exists():
    NEXT = 'teacher_v2'
    ds = {VARIANTS['gray'][0]: (VARIANTS['gray'][1], MASKS, True)}     # pick the background the team chose
    if BIN.exists(): ds['bin'] = (BIN, None, False)
    fb = next(FEEDBACK.rglob('feedback.csv')).parent if FEEDBACK.exists() else None
    t2, res2 = tt.run_update(ds, OUT, NEXT, fb, N_AUG, extractor=ext)
    display(res2)
else:
    print('No dataset_bin.zip / feedback.zip yet: skipping the update cycle.')

In [ ]:
# ---- 9. Download outputs (Colab deletes files when the session ends) ----
!cd /content && zip -qr teacher_out.zip teacher_out -x 'teacher_out/cache/*'
!ls -lh /content/teacher_out
from google.colab import files
files.download('/content/teacher_out.zip')